# DMRGT Phase 2 — Kaggle Training Notebook

**Spec:** §7 of `DMRGT-Phase2-Build-Spec.md`.

Thin wrapper. All real logic lives in `model.py` and `train.py`, attached as the `dmrgt-code` dataset.

**Always run with Save Version → Save & Run All (Commit).** Interactive sessions die on idle and lose `/kaggle/working/`.

**Session settings required:**
- Accelerator: **GPU T4** (P100 is sm_60 and unsupported by this PyTorch build — Cell 1 asserts this)
- Internet: **OFF** (Rule 9 — data comes only from the attached dataset)
- Inputs: **`dmrgt-cache`** and **`dmrgt-code`**, both at their latest versions

**Cells:**
0. PyG availability check (informational — Phase 2 uses a hand-rolled R-GCN)
1. Environment check — dataset mounts, 4 cache files, GPU >= sm_70
2. Cache integrity — SHA-256 of all 4 parquets against `hashes.json`
3. Train all 5 models in one session
4. Verify outputs and print the manifest

> **All 5 models train in one committed run.** That guarantees identical GPU, torch build and dataset version across every model — required by Rule 13, which says only the graph may differ between the control and the graph models.

## Cell 0 — PyTorch Geometric availability (informational)

Phase 2 deliberately uses a hand-rolled `RelationMessagePassing` layer rather than PyG's `RGCNConv`. With 40 nodes and 640 edges the performance difference is negligible, and avoiding `torch-scatter` removes a compiled dependency pinned to exact torch/CUDA versions.

This cell is diagnostic only. Training does not depend on its result.

In [ ]:
import torch

print(f"torch: {torch.__version__}")
try:
    import torch_geometric
    print(f"PyG present: {torch_geometric.__version__} (not used — hand-rolled R-GCN)")
except ImportError:
    print("PyG absent — expected. Phase 2 uses RelationMessagePassing in model.py.")

## Cell 1 — Environment check

Resolves the `dmrgt-code` mount, puts it on `sys.path`, then verifies all four cache files plus the config are present and the GPU is usable.

`torch.cuda.is_available()` returns `True` even on an unsupported card, so the compute-capability assertion is required — it is what catches a P100 session.

In [ ]:
import os, sys, json
from pathlib import Path


def _find_dataset(slug: str) -> str:
    """Locate an attached Kaggle dataset across known mount layouts.

    Observed layouts:
      /kaggle/input/<slug>
      /kaggle/input/datasets/<slug>
      /kaggle/input/datasets/<username>/<slug>   <- current
    """
    root = Path("/kaggle/input")
    for c in (root / slug, root / "datasets" / slug):
        if c.is_dir():
            return str(c)
    matches = [p for p in root.rglob(slug) if p.is_dir()]
    if len(matches) == 1:
        return str(matches[0])
    raise FileNotFoundError(
        f"Dataset '{slug}' not found (matches: {len(matches)}). "
        f"Attach it, then RESTART THE SESSION — attaching a dataset does not "
        f"mount it into an already-running kernel.\n"
        f"Contents of /kaggle/input:\n  "
        + "\n  ".join(sorted(str(p) for p in root.rglob("*"))[:40])
    )


sys.path.insert(0, _find_dataset("dmrgt-code"))

import torch
import paths

print(json.dumps(paths.describe(), indent=2))

# All four cache files plus the config must exist.
for label, p in [("prices",   paths.PRICES),
                 ("features", paths.FEATURES),
                 ("targets",  paths.TARGETS),
                 ("graphs",   paths.GRAPHS),
                 ("hashes",   paths.HASHES_JSON),
                 ("config",   paths.CONFIG)]:
    assert p.exists(), f"{label} not found at {p}"
print("[env] all 4 cache files + config present")

assert torch.cuda.is_available(), \
    "Enable the GPU: Settings -> Accelerator -> GPU T4"

major, minor = torch.cuda.get_device_capability(0)
gpu = torch.cuda.get_device_name(0)
print(f"[env] GPU: {gpu} (sm_{major}{minor})")
assert major >= 7, (
    f"{gpu} has CUDA capability sm_{major}{minor}, which this PyTorch build "
    f"does not support (requires sm_70+). Switch Accelerator to GPU T4 and "
    f"restart the session."
)

## Cell 2 — Cache integrity check

Recomputes the SHA-256 of every parquet and compares against `hashes.json`. A mismatch means a stale dataset version is attached; refusing to train is the only safe response.

Delegates to `train.verify_cache_hashes` so the notebook and the trainer cannot disagree about what "verified" means. Non-file keys such as `generated_at` are skipped by that function.

**Expect 4 hashes verified.** If it reports 3, either `paths.CACHE_PARQUETS` is missing the graphs entry or an older cache version is attached.

In [ ]:
import train

hashes = train.verify_cache_hashes(stop_on_mismatch=True)

assert len(hashes) == len(paths.CACHE_PARQUETS), (
    f"Verified {len(hashes)} files, expected {len(paths.CACHE_PARQUETS)}. "
    f"A stale dmrgt-cache version may be attached."
)
print(f"[cache] {len(hashes)}/{len(paths.CACHE_PARQUETS)} parquet hashes verified")

## Cell 3 — Train all 5 models

Each model writes `predictions_{model_id}.parquet` to `/kaggle/working/`.

| Model | Role |
|---|---|
| `transformer_nograph` | **The control** — same encoder, no graph |
| `lstm` | Second no-graph baseline; a fair-fight architecture check |
| `dmrgt_pos_only` | Ablation — do negative edges earn their place? |
| `dmrgt_no_sector` | Ablation — do sector edges earn their place? |
| `dmrgt_all3` | The model under test |

The control trains first, so a setup problem surfaces on the model whose expected result is already known (rank IC near zero, per Phase 1).

Each model is wrapped in try/except: one failure does not discard the models that already trained. The final assertion catches any that are missing.

**Expected runtime on a T4:** roughly 35–45 minutes for 100 runs (5 models x 4 folds x 5 seeds).

In [ ]:
import time
import traceback

cfg = train.load_config()   # resolves via paths.CONFIG

model_ids = list(cfg["models"]["ids"])
n_folds = len(cfg["folds"])
n_seeds = len(cfg["training"]["seeds"])

print(f"[env] {len(cfg['universe']['tickers'])} tickers | "
      f"{n_folds} folds | {n_seeds} seeds | {len(model_ids)} models")
print(f"[env] total training runs: {len(model_ids) * n_folds * n_seeds}")
print(f"[env] models: {model_ids}")

t0 = time.time()
succeeded, failed = [], []

for i, mid in enumerate(model_ids, start=1):
    print(f"\n{'=' * 64}")
    print(f"[train] MODEL {i}/{len(model_ids)}: {mid}")
    print(f"{'=' * 64}")
    try:
        train.run_all(cfg, verify_hashes=False, hashes=hashes, model_id=mid)
        succeeded.append(mid)
        print(f"[train] {mid} OK — elapsed {(time.time() - t0) / 60:.1f} min")
    except Exception:
        failed.append(mid)
        print(f"[train] {mid} FAILED — continuing with remaining models")
        traceback.print_exc()

print(f"\n{'=' * 64}")
print(f"[train] complete in {(time.time() - t0) / 60:.1f} min")
print(f"[train] succeeded: {succeeded}")
if failed:
    print(f"[train] FAILED:    {failed}")
print(f"{'=' * 64}")

assert not failed, f"These models failed and must be re-run: {failed}"

## Cell 4 — Verify outputs before the session ends

Confirms every expected file was written and prints the manifest headline. Anything missing here is lost when the session is reclaimed, so this check runs before you leave.

In [ ]:
import json
from pathlib import Path

out = Path("/kaggle/working")
cfg = train.load_config()
model_ids = list(cfg["models"]["ids"])

missing = []

print("Prediction files:")
for mid in model_ids:
    p = out / f"predictions_{mid}.parquet"
    if p.exists():
        print(f"  [ok]      predictions_{mid}.parquet  ({p.stat().st_size:,} bytes)")
    else:
        print(f"  [MISSING] predictions_{mid}.parquet")
        missing.append(p.name)

print("\nOther outputs:")
for p in sorted(out.glob("*.json")):
    print(f"  [ok]      {p.name}  ({p.stat().st_size:,} bytes)")

manifest_path = out / "run_manifest.json"
if manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    print("\nRun manifest:")
    print(f"  timestamp : {m.get('timestamp_utc')}")
    print(f"  gpu       : {m.get('gpu_name')}")
    print(f"  torch     : {m.get('torch')}")
    print(f"  models    : {m.get('models', m.get('model_id', '?'))}")
    print(f"  cache hash: {list(m.get('cache_hashes', {}).keys())}")
else:
    print("\n  [MISSING] run_manifest.json")
    missing.append("run_manifest.json")

print()
if missing:
    print(f"INCOMPLETE — missing: {missing}")
else:
    print("All outputs present.")
    print()
    print("Next steps:")
    print("  1. Save Version -> Save & Run All (Commit) if you have not already")
    print("  2. Download all 5 predictions_*.parquet and run_manifest.json")
    print("     from the committed version's Output tab")
    print("  3. Place them in artifacts/ locally")
    print("  4. Run: python compare.py")

assert not missing, f"Missing outputs: {missing}"